# Ce notebook est destiné au préprocessing. les code utilisent les utilitaires definis dans stanford_dogs_utils.

# 1.Configuration générale

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
src_root = "/content/drive/MyDrive/ml_dogs/project/src"

import sys
sys.path.append(src_root)

# Import du fichier utilitaires contenant toutes les classes et fonctions
from stanford_dogs_utils import (
    StanfordDogsProcessor,
    run_pipeline,
    show_pipeline_graph,
    save_processor_data
)

In [ ]:
# chemins de sortie
output_directory = "/content/drive/MyDrive/ml_dogs/project/output2"

# 2.Charger des données et pipeline intégré de traitement

En raison de la taille énorme des données il a fallu faire un pipeline qui intègre à la fois chargement et traitement des données (conformement à l'EDA) optimisé avec le cacul en parallèlle et par lot pour un traitement rapide.  
Pour des raisons de limites de puissance de calcul constaté lors de l modélisation, nous avons constitués deux types de donée : un échantillon de 10 races et la base de données entière pour le train et le test. les test de modélisation se feront sur l'échantillon de 10 races et le meilleur modèle sera évaluer sur les 120 races si possible.  
Il est également important de noter que les traitements faits n'étant pas spécifiques au jeu de train (cropping, standardistion, luminosité, constraste et redimensionnment) on n'a pas besoin de fitter le pipeline sur les données de train ce qui aurait d'ailleurs été très couteux. Ainsi le pipeline de traitement disponible dans le fichier pipeline.py sera utiliser lors de la prédiction à travers api.  

In [ ]:
# Definition du chemin racine vers le dataset Stanford Dogs
DATA_ROOT = "/content/drive/MyDrive/ml_dogs/project/data"

# Execution du pipeline principal en mode echantillon (10 races)
print("=== EXECUTION DU PIPELINE EN MODE ECHANTILLON ===")
processor, train_ds, test_ds = run_pipeline(
    data_root=DATA_ROOT,      # Chemin vers le dataset
    use_sample=True,          # Mode echantillon active
    num_breeds=10,            # Nombre de races a traiter
    img_size=256              # Taille d'image cible
)

Output hidden; open in https://colab.research.google.com to view.

# 3. Constitution des bases de données train et test

In [ ]:
# Creation des datasets pour l'échantillon
train_ds_sample = processor.create_dataset('train', batch_size=64)
test_ds_sample = processor.create_dataset('test', batch_size=32)


Creation du dataset TRAIN...
Dataset cree en 0.07s
Taille de batch: 64

Creation du dataset TEST...
Dataset cree en 0.05s
Taille de batch: 32


In [ ]:
# Sauvegarder les donnees échantillon
save_processor_data(processor, output_directory, mode_suffix="sample")

Objet processor sauvegarde a: /content/drive/MyDrive/ml_dogs/project/output2/processor_object_sample.pkl
Train data saved to: /content/drive/MyDrive/ml_dogs/project/output2/data_train/train_data_sample.pkl
Test data saved to: /content/drive/MyDrive/ml_dogs/project/output2/data_test/test_data_sample.pkl


In [ ]:
# Basculement vers le mode complet (toutes les 120 races)
print("\n=== BASCULEMENT VERS LE MODE COMPLET ===")
processor.switch_mode(use_sample=False)

train_ds_full = processor.create_dataset('train', batch_size=64)
test_ds_full = processor.create_dataset('test', batch_size=32)



=== BASCULEMENT VERS LE MODE COMPLET ===
Mode change: Complet

Chargement des donnees...
Generation des donnees...
Traitement des fichiers d'entrainement...


Traitement: 100%|██████████| 12000/12000 [07:50<00:00, 25.50it/s]


Traitement des fichiers de test...


Traitement: 100%|██████████| 8580/8580 [05:30<00:00, 25.98it/s]


Donnees chargees en 802.36s

Statistiques:
   Races: 120
   Entrainement: 12000
   Test: 8580
   Total: 20580

Creation du dataset TRAIN...
Dataset cree en 0.27s
Taille de batch: 64

Creation du dataset TEST...
Dataset cree en 0.18s
Taille de batch: 32


In [ ]:
# Affichage du diagramme du pipeline de traitement
print("\n=== DIAGRAMME DU PIPELINE ===")
pipeline_diagram = show_pipeline_graph()
pipeline_diagram  # Affichage direct dans le notebook


=== DIAGRAMME DU PIPELINE ===


In [ ]:
# Sauvegarde des donnees traitees dans des fichiers pickle
print("\n=== SAUVEGARDE DES DONNEES ===")

# Sauvegarder les donnees du mode complet
save_processor_data(processor, output_directory, mode_suffix="full")


=== SAUVEGARDE DES DONNEES ===
Objet processor sauvegarde a: /content/drive/MyDrive/ml_dogs/project/output2/processor_object_full.pkl
Train data saved to: /content/drive/MyDrive/ml_dogs/project/output2/data_train/train_data_full.pkl
Test data saved to: /content/drive/MyDrive/ml_dogs/project/output2/data_test/test_data_full.pkl


In [ ]:
# Exemples d'utilisation supplementaires :

# Pour creer un nouveau processeur avec parametres specifiques :
# processor_custom = StanfordDogsProcessor(
#     data_root=DATA_ROOT,
#     img_size=224,        # Taille compatible avec les modeles pre-entraines
#     use_sample=True,
#     num_breeds=20        # Plus de races que l'exemple initial
# )

# Pour changer le nombre de races en mode echantillon :
# processor.switch_mode(use_sample=True, num_breeds=20)
# train_ds_20 = processor.create_dataset('train', batch_size=128)

# Pour benchmark un dataset specifique :
# speed = processor.benchmark(train_ds_full, "Dataset Complet", num_batches=10)

# Pour visualiser des echantillons :
# processor.visualize(train_ds, num_samples=8)